### Databricks notebook source

Silver — nettoyage et fiabilisation des retours
Même logique que `silver_ventes.py`, avec une règle supplémentaire propre aux retours : la détection des lignes dont `id_vente` ne correspond à aucune vente existante (retours "orphelins").


In [0]:
%run "../libs/data_quality"

In [0]:
from pyspark.sql import functions as F

In [0]:
df_bronze = spark.table("bronze.ventes.retours")
print(f"Lignes en entrée (bronze) : {df_bronze.count()}")

In [0]:
# MAGIC ## Règle 1 — Cast sécurisé de la date de retour

df, rejets_date = safe_cast_date(df_bronze, "date_retour")
log_quality_summary("Cast date_retour", df, rejets_date)

In [0]:
# MAGIC ## Règle 2 — Retours orphelins
# MAGIC 
# MAGIC On s'appuie sur la table silver des ventes déjà nettoyée — pas bronze —
# MAGIC pour cette vérification. Pourquoi : un retour dont l'`id_vente` existe dans
# MAGIC bronze mais dont la ligne de vente correspondante a été rejetée en silver
# MAGIC (date invalide, champ critique manquant...) n'a plus de vente "valide" à
# MAGIC rattacher. Comparer contre silver, pas bronze, capture aussi ce cas.

df_ventes_valides = spark.table("silver.ventes.ventes_clean").select("id_vente")

df, retours_orphelins = flag_orphan_records(df, df_ventes_valides, join_key="id_vente")
log_quality_summary("Retours orphelins", df, retours_orphelins)

In [0]:
# MAGIC ## Règle 3 — Cohérence temporelle
# MAGIC 
# MAGIC Un retour ne peut pas avoir lieu avant la vente qui l'a déclenché. On sait
# MAGIC avoir injecté ce défaut volontairement. Cette règle nécessite une jointure
# MAGIC avec la date de vente — assez spécifique pour rester ici plutôt que dans
# MAGIC `data_quality.py`, contrairement aux règles génériques réutilisables.

# COMMAND ----------

df_dates_ventes = spark.table("silver.ventes.ventes_clean").select(
    "id_vente", F.col("date_vente").alias("date_vente_ref")
)

df_avec_date_vente = df.join(df_dates_ventes, on="id_vente", how="left")

incoherence_temporelle = F.col("date_retour") < F.col("date_vente_ref")

retours_incoherents = (
    df_avec_date_vente
    .filter(incoherence_temporelle)
    .drop("date_vente_ref")
)
df = (
    df_avec_date_vente
    .filter(~incoherence_temporelle)
    .drop("date_vente_ref")
)

log_quality_summary("Cohérence temporelle", df, retours_incoherents)

In [0]:
# MAGIC ## Doublons exacts

# COMMAND ----------

n_avant = df.count()
df = remove_exact_duplicates(df)
n_apres = df.count()
print(f"Doublons supprimés : {n_avant - n_apres}")

In [0]:
# MAGIC ## Assemblage de la table de rejets consolidée

# COMMAND ----------

colonnes_rejet = ["id_retour", "id_vente", "date_retour", "motif_retour", "statut"]

def normaliser_rejet(df_rejet, motif: str):
    return (
        df_rejet
        .withColumn("date_retour", F.col("date_retour").cast("string"))
        .select(*colonnes_rejet)
        .withColumn("motif_rejet", F.lit(motif))
    )

df_rejets = (
    normaliser_rejet(rejets_date, "date_invalide")
    .unionByName(normaliser_rejet(retours_orphelins, "vente_introuvable"))
    .unionByName(normaliser_rejet(retours_incoherents, "date_retour_anterieure_vente"))
    .withColumn("_rejected_at", F.current_timestamp())
)

print(f"Total lignes en quarantaine : {df_rejets.count()}")
display(df_rejets.groupBy("motif_rejet").count())

In [0]:
# MAGIC ## Écriture des tables silver

# COMMAND ----------

(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.ventes.retours_clean")
)

(
    df_rejets.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.ventes.retours_rejected")
)

print(f"silver.ventes.retours_clean : {df.count()} lignes")
print(f"silver.ventes.retours_rejected : {df_rejets.count()} lignes")

In [0]:
%sql
SELECT * FROM silver.ventes.retours_clean LIMIT 5;

In [0]:
%sql
SELECT motif_rejet, COUNT(*) AS nb
FROM silver.ventes.retours_rejected
GROUP BY motif_rejet;